In [ ]:
import yfinance as yf
import pandas as pd
import datetime
import os

# -----------------------------------------------------
# 1) Récupération de la liste complète des tickers US
# -----------------------------------------------------

# Récupérer tous les tickers disponibles
tickers = yf.Tickers("").tickers

# Convertir en liste simple
ticker_list = list(tickers.keys())

print(f"Nombre total de tickers trouvés : {len(ticker_list)}")

# -----------------------------------------------------
# 2) Téléchargement des données
# -----------------------------------------------------

# Dossier de sauvegarde
output_folder = "stocks_data"
os.makedirs(output_folder, exist_ok=True)

# Intervalle de téléchargement
start_date = "2020-01-01"
end_date = datetime.date.today().strftime("%Y-%m-%d")

for ticker in ticker_list:
    try:
        print(f"Téléchargement : {ticker}")
        data = yf.download(ticker, start=start_date, end=end_date)

        # Sauter les actions sans données
        if data.empty:
            print(f"⚠️ Pas de données pour {ticker}")
            continue

        # Sauvegarde dans un fichier Excel
        file_path = os.path.join(output_folder, f"{ticker}.xlsx")
        data.to_excel(file_path)

    except Exception as e:
        print(f"Erreur pour {ticker} : {e}")

print("Téléchargement terminé !")

In [ ]:
# Télécharger "toutes" les actions US (NASDAQ + NYSE/AMEX via Nasdaq Trader) depuis 2020-01-01

output_folder = "stocks_data"
os.makedirs(output_folder, exist_ok=True)

start_date = "2020-01-01"
end_date = datetime.date.today().strftime("%Y-%m-%d")

# 1) Construire la liste des symboles
if not tickers:  # tickers est actuellement {}
    nasdaq_url = "https://www.nasdaqtrader.com/dynamic/symdir/nasdaqlisted.txt"
    other_url = "https://www.nasdaqtrader.com/dynamic/symdir/otherlisted.txt"

    df_nasdaq = pd.read_csv(nasdaq_url, sep="|")
    df_other = pd.read_csv(other_url, sep="|")

    symbols_nasdaq = df_nasdaq.loc[df_nasdaq["Symbol"] != "File Creation Time", "Symbol"].dropna().astype(str)
    symbols_other = df_other.loc[df_other["ACT Symbol"] != "File Creation Time", "ACT Symbol"].dropna().astype(str)

    all_symbols = pd.concat([symbols_nasdaq, symbols_other], ignore_index=True)
    all_symbols = (
        all_symbols.str.strip()
        .str.upper()
        .loc[lambda s: ~s.str.contains(r"[\^\$]", regex=True)]  # filtre symboles non standards
        .drop_duplicates()
        .tolist()
    )
else:
    # Si tickers contient déjà des symboles, on les utilise
    all_symbols = sorted({str(s).strip().upper() for s in tickers if str(s).strip()})

print(f"Nombre de symboles à télécharger : {len(all_symbols)}")

# 2) Télécharger un par un
ok, skipped, failed = 0, 0, 0

for i, symbol in enumerate(all_symbols, 1):
    try:
        print(f"[{i}/{len(all_symbols)}] {symbol}")
        data = yf.download(symbol, start=start_date, end=end_date, progress=False, threads=False)

        if data.empty:
            skipped += 1
            continue

        safe_symbol = symbol.replace("/", "-")
        file_path = os.path.join(output_folder, f"{safe_symbol}.xlsx")
        data.to_excel(file_path)
        ok += 1

    except Exception as e:
        failed += 1
        print(f"Erreur pour {symbol}: {e}")

print(f"Terminé. OK={ok}, Sans données={skipped}, Erreurs={failed}")

In [5]:
import yfinance as yf
import pandas as pd
import datetime
import os


output_folder = "stocks_data"
os.makedirs(output_folder, exist_ok=True)

start_date = "2020-01-01"
end_date = datetime.date.today().strftime("%Y-%m-%d")

specific_tickers = [
    "AAPL", "MSFT", "GOOGL", "NVDA", "META", "AVGO", "TSLA", "ORCL", "AMD", "ADBE",
    "CRM", "INTC", "CSCO", "IBM", "QCOM", "TXN", "NOW", "AMAT", "MU", "LRCX",
    "PLTR", "DELL", "NET", "JPM", "BAC", "WFC", "GS", "MS", "C", "BLK",
    "SCHW", "AXP", "USB", "PNC", "COF", "JNJ", "UNH", "LLY", "PFE", "ABBV",
    "MRK", "TMO", "ABT", "DHR", "BMY", "AMGN", "GILD", "AMZN", "WMT", "COST",
    "HD", "TGT", "LOW", "MCD", "SBUX", "NKE", "KO", "PEP", "PG", "DG",
    "BKNG", "XOM", "CVX", "COP", "SLB", "EOG", "PXD", "OXY", "MPC", "PSX",
    "VLO", "HAL", "GE", "BA", "CAT", "HON", "RTX", "LMT", "UPS", "UNP",
    "DE", "MMM", "NOC", "VZ", "T", "TMUS", "CMCSA", "DIS", "NFLX", "CHTR",
    "AMT", "PLD", "EQIX", "SPG", "PSA", "O", "DLR", "SPY", "QQQ", "IVV",
    "VOO", "VTI", "DIA", "IWM", "VGT", "XLK", "XLF", "XLE", "XLV", "GLD",
    "SLV", "EEM", "VEA", "BABA", "TSM", "NVO", "ASML", "SAP", "TM", "SONY",
    "BHP", "RIO", "UL", "NVS", "BP", "SHEL", "COIN", "MSTR", "MARA", "RIOT", "CLSK"
]

os.makedirs(output_folder, exist_ok=True)
ok, skipped, failed = 0, 0, 0

for i, symbol in enumerate(specific_tickers, 1):
    try:
        print(f"[{i}/{len(specific_tickers)}] Téléchargement : {symbol}")
        data = yf.download(symbol, start=start_date, end=end_date, progress=False, threads=False)

        if data.empty:
            print(f"⚠️ Pas de données pour {symbol}")
            skipped += 1
            continue

        safe_symbol = symbol.replace("/", "-")
        file_path = os.path.join(output_folder, f"{safe_symbol}.xlsx")
        data.to_excel(file_path)
        ok += 1
        print(f"✅ {symbol} sauvegardé")

    except Exception as e:
        failed += 1
        print(f"❌ Erreur pour {symbol} : {e}")

print(f"\nTerminé. ✅ OK={ok} | ⚠️ Sans données={skipped} | ❌ Erreurs={failed}")

[1/131] Téléchargement : AAPL


C:\Users\DYLANE\AppData\Local\Temp\ipykernel_9984\1474148120.py:35: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start=start_date, end=end_date, progress=False, threads=False)


✅ AAPL sauvegardé
[2/131] Téléchargement : MSFT


C:\Users\DYLANE\AppData\Local\Temp\ipykernel_9984\1474148120.py:35: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start=start_date, end=end_date, progress=False, threads=False)


✅ MSFT sauvegardé
[3/131] Téléchargement : GOOGL


...

✅ MARA sauvegardé
[130/131] Téléchargement : RIOT


C:\Users\DYLANE\AppData\Local\Temp\ipykernel_9984\1474148120.py:35: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start=start_date, end=end_date, progress=False, threads=False)


✅ RIOT sauvegardé
[131/131] Téléchargement : CLSK


C:\Users\DYLANE\AppData\Local\Temp\ipykernel_9984\1474148120.py:35: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start=start_date, end=end_date, progress=False, threads=False)


✅ CLSK sauvegardé

Terminé. ✅ OK=130 | ⚠️ Sans données=1 | ❌ Erreurs=0


In [ ]:
# -----------------------------------------------------
# Chargement et consolidation de tous les fichiers Excel
# -----------------------------------------------------

all_data = {}

for file in os.listdir(output_folder):
    if file.endswith(".xlsx"):
        ticker_name = file.replace(".xlsx", "")
        try:
            df = pd.read_excel(os.path.join(output_folder, file), index_col=0, header=[0, 1])
            # Aplatir les colonnes MultiIndex si nécessaire
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [col[0] for col in df.columns]
            df.index = pd.to_datetime(df.index)
            all_data[ticker_name] = df
        except Exception as e:
            print(f"❌ Erreur lors du chargement de {file} : {e}")

print(f"✅ {len(all_data)} fichiers chargés avec succès.")

# Extraire uniquement les prix de clôture pour tous les tickers
close_prices = pd.DataFrame({
    ticker: df["Close"]
    for ticker, df in all_data.items()
    if "Close" in df.columns
})

close_prices.index = pd.to_datetime(close_prices.index)
close_prices.sort_index(inplace=True)

print(f"\n📊 Tableau des prix de clôture : {close_prices.shape[0]} jours × {close_prices.shape[1]} tickers")
print(close_prices.tail())